# All Tests

```python
One sample mean |z, t
normality check | Shapiro wilk test
not normal | Mann Whitney U test
variance check | Levene / Bartlette test
two independent sample means | z, t, Welch's t
two related sample means | z, t
one sample proportion | z
two sample proportion | z
two sample variance | F
> two sample variance, one category | one way ANOVA (F)
pair-wise comparision | Tukey Cramer 
more than two sample variance, two category | two way ANOVA (F)
two sample proportion, two categories | chi-squared
> two sample proportion, two categories | chi-squared
pair-wise comparision | Marascuilo Procedure
> two sample proportion, > two categories | chi-squared
```

![ab_test_tree.jpeg](./ab_test_tree.jpeg "ab_test_tree.jpeg")

%md
| Scenario                                                | Statistical Test                                              | Key Assumptions                                                                                                                             | If Assumptions Fail                                  |
| ------------------------------------------------------- | ------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------- |
| **One sample mean**                                     | Z-test (σ known), One-sample t-test (σ unknown)               | Random sample; population approximately normal (or n ≥ 30); independent observations                                                        | Wilcoxon Signed-Rank Test                            |
| **Normality check**                                     | Shapiro-Wilk Test                                             | Continuous data; random sample                                                                                                              | If p < 0.05, assume non-normality                    |
| **Non-normal comparison (2 independent groups)**        | Mann-Whitney U Test                                           | Independent samples; ordinal/continuous data; similar distribution shapes                                                                   | —                                                    |
| **Equal variance check**                                | Levene's Test (preferred), Bartlett's Test (normal data only) | Independent groups; Bartlett assumes normality                                                                                              | Use Welch's t-test or Welch's ANOVA                  |
| **Two independent sample means**                        | Z-test, Independent t-test, Welch's t-test                   | Independent samples; normal populations (or large sample); equal variances for Student's t-test; Welch does **not** require equal variances | Mann-Whitney U Test                                  |
| **Two related sample means**                            | Paired t-test                                                 | Paired observations; differences approximately normal                                                                                       | Wilcoxon Signed-Rank Test                            |
| **One sample proportion**                               | One-sample Z-test for proportion                              | Random sample; (np >= 5), (n(1-p) >= 5)                                                                                                   | Exact Binomial Test                                  |
| **Two sample proportions**                              | Two-proportion Z-test                                         | Independent samples; expected successes and failures ≥ 5 in each group                                                                      | Fisher's Exact Test (small samples)                  |
| **Compare two variances**                               | F-test                                                        | Independent samples; both populations normally distributed                                                                                  | Levene's Test or Brown-Forsythe Test                 |
| **Compare >2 means (one categorical factor)**           | One-Way ANOVA (F-test)                                        | Independent observations; normal residuals; equal variances                                                                                 | Kruskal-Wallis Test                                  |
| **Pairwise comparison after One-Way ANOVA**             | Tukey HSD (Tukey-Kramer if unequal sample sizes)              | ANOVA assumptions satisfied                                                                                                                 | Dunn's Test (after Kruskal-Wallis)                   |
| **Compare >2 means (two categorical factors)**          | Two-Way ANOVA                                                 | Independent observations; normal residuals; equal variances; interaction effects modeled                                                    | Aligned Rank Transform ANOVA / Scheirer-Ray-Hare     |
| **Association between two categorical variables**       | Chi-Square Test of Independence                               | Independent observations; expected cell count ≥ 5                                                                                           | Fisher's Exact Test                                  |
| **Pairwise comparison of proportions after Chi-Square** | Marascuilo Procedure                                          | Significant Chi-Square result; independent samples                                                                                          | Pairwise proportion tests with Bonferroni correction |
| **Proportions across >2 categories/groups**             | Chi-Square Test                                               | Independent observations; expected counts ≥ 5                                                                                               | Fisher-Freeman-Halton Exact Test (small samples)     |


# Causal modeling
- [Causal Machine Learning](https://www.geeksforgeeks.org/machine-learning/causal-machine-learning/)

In [0]:
!pip install git+https://github.com/microsoft/dowhy.git -q

In [0]:
import numpy as np
import pandas as pd
from dowhy import CausalModel
import dowhy.datasets
import statsmodels.api as sm

In [0]:
data_dict = dowhy.datasets.linear_dataset(
    beta=2,
    num_common_causes=2,
    num_discrete_common_causes=1,
    num_instruments=1,
    num_samples=5000,
    treatment_is_binary=True
)
data_df = data_dict["df"]

In [0]:
print(data_df.columns.tolist())
data_df.head()

| Variable | Data Type | Role in Causal Model | Notes |
|----------|-----------|----------------------|-------|
| `y`      | float64   | Outcome              | Dependent variable we want to explain or predict |
| `v0`     | bool      | Treatment            | Binary variable whose causal effect on `y` is estimated |
| `Z0`     | float64   | Instrument           | Affects `v0` but not `y` directly (exclusion restriction) |
| `W0`     | float64   | Confounder           | Observed covariate to control for confounding |
| `W1`     | category  | Confounder           | Categorical covariate to control for confounding |


In [0]:
causal_graph = """
digraph {
    Treatment -> Outcome;       y
    Confounder1 -> Treatment;   v0
    Confounder2 -> Treatment;
    Instrument1 -> Treatment;   Z0
    Confounder1 -> Outcome;     W0
    Confounder2 -> Outcome;     W1
}
"""

causal_model = CausalModel(
    data=data_df,
    treatment=data_dict["treatment_name"],
    outcome=data_dict["outcome_name"],
    graph=data_dict["gml_graph"]
)

In [0]:
X = sm.add_constant(data_df[data_dict["treatment_name"]].astype(float))
Y = data_df[data_dict["outcome_name"]].astype(float)
ols_model = sm.OLS(Y, X).fit()
print(ols_model.summary().tables[1])

In [0]:
identified_estimand = causal_model.identify_effect()
print(identified_estimand)

In [0]:
causal_estimate = causal_model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting"
)
print(causal_estimate)

In [0]:
refutation_test = causal_model.refute_estimate(
    identified_estimand,
    causal_estimate,
    method_name="add_unobserved_common_cause"
)
print(refutation_test)

In [0]:
iv_estimate = causal_model.estimate_effect(
    identified_estimand,
    method_name="iv.instrumental_variable"
)
print(iv_estimate)